In [ ]:
!nvidia-smi


In [ ]:
!pip install whisperx

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
AUDIO_DIR  = "/content/drive/MyDrive/audio/self_help"
OUTPUT_DIR = "/content/drive/MyDrive/Transcripts_CSS/transcripts_motivational"
DEVICE     = "cuda"
BATCH_SIZE = 16


In [ ]:
#corruption check
import json
import glob
import os


def remove_error_jsons(folder):
    removed = []

    for path in glob.glob(os.path.join(folder, "*.json")):
        try:
            with open(path, encoding="utf-8") as f:
                data = json.load(f)

            # Remove any JSON file that contains an "error" field
            if isinstance(data, dict) and "error" in data:
                os.remove(path)
                removed.append(os.path.basename(path))

        except json.JSONDecodeError:
            # Also remove malformed JSON files
            os.remove(path)
            removed.append(os.path.basename(path))

    print(f"Removed {len(removed)} files.")
    for f in removed:
        print(f"  {f}")

# Use your folder variable
remove_error_jsons("/content/drive/MyDrive/transcripts_motivational")



In [ ]:
#load models & GPU warmup
import whisperx
import numpy as np
import torch

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading Whisper model...")
whisper_model = whisperx.load_model("large-v2", DEVICE, compute_type="float16")

print("Loading alignment model...")
align_model, metadata = whisperx.load_align_model(language_code="hi", device=DEVICE)

# GPU warmup — prevents CUDA errors on first real file
print("Warming up GPU...")
dummy = np.zeros(16000, dtype=np.float32)
whisper_model.transcribe(dummy, batch_size=BATCH_SIZE)
print("Ready")


In [ ]:
#transcription helpers
import time

def transcribe_with_retry(path, retries=3, wait=10):
    for attempt in range(retries):
        try:
            audio = whisperx.load_audio(path)
            raw = whisper_model.transcribe(audio, batch_size=BATCH_SIZE)
            aligned = whisperx.align(
                raw["segments"], align_model, metadata, audio, DEVICE
            )
            return aligned, raw.get("language", "")
        except RuntimeError as e:
            if "cuda" in str(e).lower() and attempt < retries - 1:
                print(f"  CUDA error (attempt {attempt+1}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

def build_output(filename, language, aligned):
    return {
        "filename": filename,
        "language": language,
        "segments": [
            {
                "start": s["start"],
                "end":   s["end"],
                "text":  s["text"],
                "words": [
                    {
                        "word":  w["word"],
                        "start": w.get("start"),
                        "end":   w.get("end"),
                        "score": w.get("score")
                    }
                    for w in s.get("words", [])
                ]
            }
            for s in aligned["segments"]
        ]
    }


In [ ]:
#main batch loop

print("-----------")
from tqdm.auto import tqdm

EXTENSIONS = ("*.wav", "*.mp3", "*.m4a", "*.flac")

audio_files = sorted(
    f for ext in EXTENSIONS
    for f in glob.glob(os.path.join(AUDIO_DIR, ext))
)

already_done = {
    os.path.basename(p).replace(".json", "")
    for p in glob.glob(OUTPUT_DIR + "/*.json")
}

pending = [
    p for p in audio_files
    if os.path.basename(p).rsplit(".", 1)[0] not in already_done
]

print(f"Total files   : {len(audio_files)}")
print(f"Already done  : {len(already_done)}")
print(f"To process    : {len(pending)}")

for path in tqdm(pending):
    fname    = os.path.basename(path)
    out_path = os.path.join(OUTPUT_DIR, fname.rsplit(".", 1)[0] + ".json")

    try:
        aligned, language = transcribe_with_retry(path)
        result = build_output(fname, language, aligned)
    except Exception as e:
        result = {"filename": fname, "error": str(e)}

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

print("Session complete.")


In [ ]:
#progress check
done   = glob.glob(OUTPUT_DIR + "/*.json")
errors = []
ok     = []

for p in done:
    with open(p, encoding="utf-8") as f:
        d = json.load(f)
    (errors if "error" in d else ok).append(os.path.basename(p))

print(f"Successfully transcribed : {len(ok)}")
print(f"Errors                   : {len(errors)}")
print(f"Remaining                : {len(audio_files) - len(done)}")

if errors:
    print("\nError files:")
    for f in errors: print(f"  {f}")
